# Football Shots Model — Colab backfill & explorationThis notebook is for two things **only**:1. The one-time initial backfill of this season's data (everything after that is handled by the weekly GitHub Actions job — see the README).2. Ad-hoc exploration: poke at the database, sanity-check a player's rolling form, look at drift charts, etc.It does **not** run the Streamlit app (Colab can't host a persistent public app) and it does **not** replace the weekly automation — it's a workbench, not the production path.

## 1. SetupClone your repo (after you've pushed the generated project to GitHub) and install dependencies.

In [ ]:
# Replace with your repo URL once you've pushed it to GitHubREPO_URL = "https://github.com/<you>/football-shots-model.git"!git clone $REPO_URL project%cd project!pip install -q -r requirements.txt

## 2. Run the initial backfillThis scrapes every completed match from `SEASON_START_DATE` (in `config.py`) up to today, across all 8 competitions. **This is the single biggest scrape the pipeline ever does** — expect it to take a while, because of the deliberate delay between requests (politeness, not a bug — see `src/scraper.py`).Rough sizing: a full matchday across all 8 competitions is on the order of 10-25 matches, so budget roughly 5-6 requests per matchday (1 for the day-discovery page + 1 per match report actually in scope), each with a several-second delay. A month of backfill is a few hundred requests — expect tens of minutes, not seconds. Run this cell, then go do something else for a while.

In [ ]:
import syssys.path.insert(0, '.')from src import pipelineimport configsummary = pipeline.scrape_and_store_completed(    date_from=config.SEASON_START_DATE,    date_to=None,  # defaults to today)summary

## 3. Train the first modelNeeds a few completed matchdays in the database before there's anything meaningful to learn from.

In [ ]:
result = pipeline.weekly_retrain()result

## 4. Sanity-check: look at a few predictions

In [ ]:
preds = pipeline.generate_weekly_predictions()preds.sort_values('pred_shots', ascending=False).head(20)

## 5. Push the backfilled data back to GitHubThis is what hands off to the automated weekly job — commit the populated `data/matches.db` and the trained `models/` files back to the repo. After this, GitHub Actions takes over and you shouldn't need to run this notebook again except to explore.

In [ ]:
!git config user.email "you@example.com"!git config user.name "Your Name"!git add data/matches.db models/!git commit -m "Initial backfill"!git push

## 6. (Optional) Free-form explorationEverything below is scratch space — the database and trained models are just files, query them however is useful.

In [ ]:
from src import dbraw = db.load_player_match_stats()raw.groupby('competition').size()

In [ ]:
# Example: a specific player's match-by-match shot log this seasonraw[raw.player_name.str.contains('', case=False)].sort_values('date')[    ['date', 'competition', 'team', 'opponent', 'minutes', 'shots', 'shots_on_target']]